# 05 — Monitoring and actionability

This notebook presents data-plan Section 6: a **monitoring blueprint** of eight
QL2 controls and their evaluation on this one-off sample. For every control it
states the condition, the severity, the likely business impact and the
recommended response, and then what the supplied evidence shows.

It is a thin presentation layer. The control catalog, every evaluation, the
tables and the summary come from `ql2_sixt_canada_analysis.monitoring`
(`run_monitoring` → `run_pricing_pipeline` → price-change analysis and
visible assortment on that same result → control evaluation). It does not
depend on earlier notebooks or kernel state.

This is not a scheduler, daemon, alerting service, dashboard or notification
integration. Nothing here raises an alert, sends a notification or writes a
file.

## Data-plan scope

Data-plan Section 6, *Monitoring and actionability*, requires these controls:

1. Missing expected locations
2. Job/detail count mismatches
3. Duplicate or aliased location feeds
4. Unexpected timestamp offsets
5. Invalid or changing product attributes
6. Abrupt assortment changes
7. Large synchronized price movements
8. Unconfirmed anomalies at the end of a collection window

The sample spans roughly 90 hours. That is enough to evaluate the
contract-based controls, which compare the extract with approved contracts,
but it is too short to calibrate any production threshold.

## Severity scale

Severity communicates business impact and response urgency. It is not a
statistical confidence level. A control's catalog severity is fixed; only two
controls can escalate their effective severity: an abrupt assortment change
when an approved rule is met while collection completeness is suspect, and an
end-of-window observation that is part of a synchronized movement.

In [ ]:
from IPython.display import display

from ql2_sixt_canada_analysis import (
    PRODUCTION_CALIBRATION_REQUIREMENTS,
    monitoring_control_table,
    monitoring_summary_lines,
    run_monitoring,
    severity_scale_table,
    status_legend_table,
)
from ql2_sixt_canada_analysis.paths import resolve_raw_data_dir

display(severity_scale_table())

## Evaluation statuses

Every control has exactly one status, and the five statuses are never merged:

- `triggered` — a defect was observed in the supplied evidence;
- `passed` — every prerequisite was available and the control held;
- `not_assessable` — prerequisite evidence was unavailable, which is never a
  pass;
- `candidate_only` — a proposed rule without an approved or calibrated
  threshold; its observations are review candidates, not alerts and not passes;
- `confirmation_required` — a right-censored observation at the end of the
  window that needs another eligible collection before confirmation.

In [ ]:
display(status_legend_table())

## Confidentiality and side effects

All QL2 data is proprietary. This notebook shows only the fixed control
catalog, enum statuses and typed finding codes. It never shows rows, product
values, prices, identifiers, source labels, file names, paths or real-data
metrics, and the summary lines contain status values and control names only.

Nothing is written, and no alert or notification is produced. Clear all
outputs before committing (Edit ▸ Clear Outputs of All Cells).

In [ ]:
# Raw-data directory: the centralized default unless the documented environment
# override is set. Set RAW_DIR_OVERRIDE only for a deliberate local experiment.
RAW_DIR_OVERRIDE = None

raw_dir = resolve_raw_data_dir(RAW_DIR_OVERRIDE)
print("Raw-data directory selected.")

## Pipeline evidence

One call runs the pricing pipeline exactly once. The price-change analysis and
the visible assortment are derived from that same result and must be bound to
its frames and location authority. If the evidence chain cannot be bound, the
report is blocked: only the blocker categories are shown, and every control is
`not_assessable` while its definition stays visible. If pricing readiness is
blocked, the structural controls are still evaluated from their own reports
and the downstream controls are `not_assessable`.

In [ ]:
result = run_monitoring(raw_dir)
report = result.report
for line in monitoring_summary_lines(report):
    print(line)

## Control definitions and sample evaluation

One row per required control, in data-plan order. The definition columns
(condition, severity, likely business impact, recommended response,
calibration status) come from the fixed catalog. The evaluation columns
(effective severity, evaluation status, observation, findings, unavailable
evidence, notes and policy status) hold only enum values and typed codes.

In [ ]:
controls = monitoring_control_table(report)
display(controls)

## Current sample evidence versus proposed operational controls

- **Current sample evidence.** Missing expected locations, job/detail count
  mismatches, duplicate or aliased location feeds, unexpected timestamp offsets
  and invalid or changing product attributes are contract-based. They compare
  the extract with approved contracts (the expected-stream contract, the
  per-job reconciliation, the location authority and the Vancouver identity
  policy, the city time zones and the product identity contract), so they are
  evaluated now and can pass or trigger.
- **Proposed operational controls.** Abrupt assortment changes and large
  synchronized price movements are candidate-only. Their methods and
  thresholds are not approved, so the evaluation lists review candidates and
  never raises a production alert or reports a pass.
- **Confirmation.** Unconfirmed anomalies at the end of a collection window
  are right-censored observations. They are `confirmation_required`: neither
  persistent nor failed.

## Candidate rules and right-censored observations

- The unusual-drop policy is unavailable. Observed drops are review
  candidates, not proven anomalies, collection failures or supplier
  withdrawals.
- No synchronized-movement parameters are approved. Synchronized movements are
  descriptive evidence; they are not proof of intentional repricing or of an
  extraction failure. Increases and decreases are kept separate, and the
  Vancouver Downtown and Thurlow alias is one canonical location, so it never
  duplicates an event.
- Right-censored events provide no persistence evidence. A final-window event
  needs another eligible collection, and missing captures and hard breaks are
  never bridged.
- Behavioral similarity between location feeds is evidence for review only and
  never establishes an alias.

## Limitations and production calibration

The thresholds of the candidate rules cannot be calibrated from roughly 90
hours of one extract: no threshold here is validated, calibrated or
statistically stable, and none was optimized against this sample. Before
production monitoring, the following is required:

In [ ]:
for requirement in PRODUCTION_CALIBRATION_REQUIREMENTS:
    print("-", requirement)

## Data-plan reconciliation

- Missing expected locations: implemented from the approved expected-stream contract, expected-location coverage and per-stream scheduled coverage; contract-based and evaluated, with governed exclusions kept separate from unexplained gaps.
- Job/detail count mismatches: implemented from the per-job reconciliation and the job linkage; contract-based and evaluated, so offsetting over-counts and under-counts cannot cancel.
- Duplicate or aliased location feeds: implemented from coverage, the location authority, the Vancouver identity policy and the canonical-offer combination; contract-based and evaluated, and behavioral similarity never establishes an alias.
- Unexpected timestamp offsets: implemented from the temporal authority, temporal reconciliation and the per-stream schedule assignment; contract-based and evaluated against the approved city time zones.
- Invalid or changing product attributes: implemented from vehicle-attribute stability; contract-based and evaluated, with missing values, insufficient history and proven conflicts kept distinct.
- Abrupt assortment changes: implemented from the visible-assortment engine; candidate-only because the unusual-drop policy is unavailable, so observed drops are review candidates and no alert is raised.
- Large synchronized price movements: implemented from the higher-order price-change analysis; candidate-only because no synchronized-movement parameters are approved, with increases and decreases kept separate.
- Unconfirmed anomalies at the end of a collection window: implemented from right-censored persistence and the assortment timeline; final-window observations are confirmation-required, never persistent, and breaks are never bridged.

## Clear all outputs before committing

Use Edit ▸ Clear Outputs of All Cells before committing. `tests/test_notebooks.py`
fails if a committed notebook has outputs or execution counts.

In [ ]:
print("No files written.")